# JOIN Predicate Column Lineage

**Example: Tracking JOIN ON Predicate Columns in Column Lineage (Gap 7)**


This example demonstrates how clgraph tracks JOIN ON predicate columns as
lineage edges. Before Gap 7, only value-flow columns (columns in SELECT)
appeared in the lineage graph. Now, columns used in JOIN ON clauses are
tracked as predicate edges, making previously invisible dependencies
visible for impact analysis.

Key features demonstrated:
1. Basic equi-join predicate edges with metadata
2. Point-in-time / range join (BETWEEN) with 5 predicate columns
3. Multi-join chain with per-join scoped predicate edges
4. Impact analysis using predicate edges with SQLColumnTracer

### Imports

In [ ]:
from clgraph import Pipeline, RecursiveLineageBuilder, SQLColumnTracer


def predicate_edges(graph):
    """Return only edges where is_join_predicate is True."""
    return [e for e in graph.edges if e.is_join_predicate]


def predicate_edges_to(graph, target):
    """Return predicate edges targeting a specific output column."""
    return [e for e in graph.edges if e.is_join_predicate and e.to_node.full_name == target]


# ============================================================
# Example 1: Basic Equi-Join Predicate Edges
# ============================================================
print("=" * 60)
print("Example 1: Basic Equi-Join Predicate Edges")
print("=" * 60)

sql_1 = """
SELECT o.order_id, o.amount, d.city AS customer_city
FROM raw_orders o
LEFT JOIN dim_customer d ON o.customer_id = d.id
"""

builder_1 = RecursiveLineageBuilder(sql_1, dialect="bigquery")
graph_1 = builder_1.build()

print(f"\nQuery:{sql_1}")
print("1a. Value edges (standard lineage):")
for edge in graph_1.edges:
    if not edge.is_join_predicate:
        print(f"  {edge.from_node.full_name} -> {edge.to_node.full_name}")

print("\n1b. Predicate edges (NEW \u2014 from JOIN ON clause):")
for edge in predicate_edges(graph_1):
    print(f"  {edge.from_node.full_name} -> {edge.to_node.full_name}")
    print(f"    \u2022 edge_type     = {edge.edge_type}")
    print(f"    \u2022 join_side     = {edge.join_side}")
    print(f"    \u2022 join_condition = {edge.join_condition}")

print("\n1c. Compare: d.city \u2192 output.customer_city is a VALUE edge:")
for edge in graph_1.edges:
    if edge.from_node.full_name == "dim_customer.city":
        print(f"  is_join_predicate = {edge.is_join_predicate}  \u2713 (value flow, not predicate)")


# ============================================================
# Example 2: Point-in-Time / Range Join (BETWEEN)
# ============================================================
print("\n" + "=" * 60)
print("Example 2: Point-in-Time Join (BETWEEN)")
print("=" * 60)

sql_2 = """
SELECT o.order_id, o.customer_id, o.order_ts, o.amount,
       d.city AS customer_city_at_order
FROM raw_orders o
LEFT JOIN dim_customer d
  ON o.customer_id = d.id
 AND o.order_ts BETWEEN d.start_time AND d.end_time
"""

builder_2 = RecursiveLineageBuilder(sql_2, dialect="bigquery")
graph_2 = builder_2.build()

print(f"\nQuery:{sql_2}")
print("2a. All predicate edges \u2192 customer_city_at_order:")
pred_edges_2 = predicate_edges_to(graph_2, "output.customer_city_at_order")
for edge in pred_edges_2:
    print(f"  {edge.from_node.full_name:30s} (join_side={edge.join_side})")

print(f"\n  \u2192 {len(pred_edges_2)} predicate columns detected")
print("  \u2192 d.start_time and d.end_time were previously INVISIBLE in lineage")

print("\n2b. Value edge is unchanged:")
for edge in graph_2.edges:
    if edge.from_node.full_name == "dim_customer.city" and not edge.is_join_predicate:
        print(f"  {edge.from_node.full_name} -> {edge.to_node.full_name}  (value edge \u2713)")


# ============================================================
# Example 3: Multi-Join Chain with Per-Join Scoping
# ============================================================
print("\n" + "=" * 60)
print("Example 3: Multi-Join Chain \u2014 Per-Join Scoping")
print("=" * 60)

sql_3 = """
SELECT a.id, b.val, c.label
FROM table_a a
INNER JOIN table_b b ON a.id = b.a_id
INNER JOIN table_c c ON b.id = c.b_id AND b.category = c.category
"""

builder_3 = RecursiveLineageBuilder(sql_3, dialect="bigquery")
graph_3 = builder_3.build()

print(f"\nQuery:{sql_3}")
print("3a. First join predicates (a.id = b.a_id) \u2192 output.val ONLY:")
for edge in predicate_edges_to(graph_3, "output.val"):
    print(f"  {edge.from_node.full_name} -> output.val")

print(
    "\n3b. Second join predicates (b.id = c.b_id AND b.category = c.category) \u2192 output.label ONLY:"
)
for edge in predicate_edges_to(graph_3, "output.label"):
    print(f"  {edge.from_node.full_name} -> output.label")

print("\n3c. No cross-join leakage \u2014 first join predicates do NOT target output.label:")
label_pred_sources = {e.from_node.full_name for e in predicate_edges_to(graph_3, "output.label")}
assert "table_a.id" not in label_pred_sources, "Cross-join leakage detected!"
assert "table_b.a_id" not in label_pred_sources, "Cross-join leakage detected!"
print("  \u2713 table_a.id NOT in output.label predicates")
print("  \u2713 table_b.a_id NOT in output.label predicates")


# ============================================================
# Example 4: Impact Analysis with Predicate Edges
# ============================================================
print("\n" + "=" * 60)
print("Example 4: Impact Analysis with Predicate Edges")
print("=" * 60)

print("\nUsing the point-in-time join from Example 2.")

print("\n4a. Forward trace from dim_customer.start_time (SQLColumnTracer):")
tracer = SQLColumnTracer(sql_2, dialect="bigquery")
forward = tracer.get_forward_lineage(["dim_customer.start_time"])
print(f"  Impacted outputs: {forward['impacted_outputs']}")
print("  \u2192 customer_city_at_order is now reachable (was invisible before Gap 7)")

print("\n4b. Filter predicate vs value edges using is_join_predicate:")
value_edges_2 = [e for e in graph_2.edges if not e.is_join_predicate]
pred_edges_all_2 = [e for e in graph_2.edges if e.is_join_predicate]
print(f"  Value edges:     {len(value_edges_2)}")
print(f"  Predicate edges: {len(pred_edges_all_2)}")
print(f"  Total edges:     {len(graph_2.edges)}")

print("\n4c. Existing value lineage is unchanged:")
for edge in value_edges_2:
    print(f"  {edge.from_node.full_name} -> {edge.to_node.full_name}")

print("\n" + "=" * 60)
print("JOIN Predicate Lineage Examples Complete!")
print("=" * 60)

### Visualize Point-in-Time Join Lineage

Display the column lineage graph for the point-in-time join (Example 2),
showing both value and predicate edges.

In [ ]:
import shutil

from clgraph import visualize_pipeline_lineage

# Create pipeline for visualization using the point-in-time join
sql_pit = """
SELECT o.order_id, o.customer_id, o.order_ts, o.amount,
       d.city AS customer_city_at_order
FROM raw_orders o
LEFT JOIN dim_customer d
  ON o.customer_id = d.id
 AND o.order_ts BETWEEN d.start_time AND d.end_time
"""
pit_pipeline = Pipeline([("pit_join", sql_pit)], dialect="bigquery")

if shutil.which("dot") is None:
    print("\u26a0\ufe0f  Graphviz not installed. Install with: brew install graphviz")
else:
    print("Point-in-Time Join \u2014 Column Lineage (value + predicate edges):")
    display(visualize_pipeline_lineage(pit_pipeline.column_graph.to_simplified()))